# FireSight-IR | Module 1a — Download VIIRS & FIRMS Data

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 1a of 9 — Data ingestion: VIIRS I4/I5 + FIRMS active fire ground truth  

---

## What this notebook does

1. Authenticates with NASA Earthdata (`.netrc` setup, one-time)  
2. Downloads **VIIRS VNP14IMG** (375 m active fire product, I4/I5-derived) for the western CONUS training domain  
3. Downloads **FIRMS active fire CSV** — the ground truth labels for all modeling stages  
4. Organises raw files into the project archive structure  
5. Runs a basic integrity check and prints a summary  

## Domain and date range

| Parameter | Value | Rationale |
|---|---|---|
| Region | Western CONUS (CA, OR, NV, AZ) | High fire frequency, gas flare false-positive source (Permian Basin edge), diverse land cover |
| Bounding box | lon −125° to −109°, lat 32° to 49° | Covers Sierra Nevada, Cascades, Mojave, Great Basin |
| Training years | 2018–2022 | 5 full fire seasons, includes 2020 Creek Fire (extreme event) and 2018 Camp Fire |
| Validation year | 2023 | Held out — never seen during training |

## Output files (after this notebook)

```
firesight-ir/
  data/
    raw/
      viirs/          ← VNP14IMG HDF5 files (one per overpass)
      firms/          ← FIRMS CSV files (one per year)
    processed/        ← populated in later notebooks
```

> **Before running:** complete Section 0 (Earthdata credentials) once. Everything else is automated.

---
## Section 0 — One-time setup: NASA Earthdata credentials

NASA Earthdata uses `.netrc`-based authentication. Run the cell below **once** on your machine. It writes `~/.netrc` and `~/.urs_cookies`. You will not need to repeat this.

**Register here first if you do not have an account:** https://urs.earthdata.nasa.gov/users/new

In [1]:
import os
import getpass
from pathlib import Path

def setup_earthdata_credentials():
    """
    Write ~/.netrc and ~/.urs_cookies for NASA Earthdata authentication.
    Run once. Credentials are stored locally — never committed to git.
    """
    netrc_path = Path.home() / '.netrc'
    cookies_path = Path.home() / '.urs_cookies'

    if netrc_path.exists():
        print(f'~/.netrc already exists at {netrc_path}')
        overwrite = input('Overwrite? (y/n): ').strip().lower()
        if overwrite != 'y':
            print('Skipping credential setup — using existing ~/.netrc')
            return

    username = input('NASA Earthdata username: ').strip()
    password = getpass.getpass('NASA Earthdata password: ')

    netrc_content = (
        f'machine urs.earthdata.nasa.gov login {username} password {password}\n'
    )
    netrc_path.write_text(netrc_content)

    # On Linux/Mac, .netrc must be owner-readable only
    try:
        os.chmod(netrc_path, 0o600)
    except Exception:
        pass  # Windows does not enforce this

    cookies_path.touch()
    print(f'Credentials written to {netrc_path}')
    print(f'Cookie jar initialised at {cookies_path}')
    print('NASA Earthdata authentication is ready.')

setup_earthdata_credentials()

NASA Earthdata username:  Msledge7
NASA Earthdata password:  ········


Credentials written to C:\Users\taylo\.netrc
Cookie jar initialised at C:\Users\taylo\.urs_cookies
NASA Earthdata authentication is ready.


---
## Section 1 — Imports and project configuration

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import time
import json
import hashlib
import logging
import urllib.request
from pathlib import Path
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Third-party ───────────────────────────────────────────────────────────────
import requests                          # pip install requests
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm           # pip install tqdm

# Optional: h5py for quick HDF5 inspection later in this notebook
try:
    import h5py
    HDF5_AVAILABLE = True
except ImportError:
    HDF5_AVAILABLE = False
    print('h5py not found — HDF5 inspection cells will be skipped.')
    print('Install with: pip install h5py')

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('firesight')

print('Imports OK')

Imports OK


In [3]:
# ── Project root — set this to wherever you cloned the repo ───────────────────
# On Windows with Anaconda, use a raw string or forward slashes:
#   PROJECT_ROOT = Path(r'C:/Users/YourName/firesight-ir')
PROJECT_ROOT = Path('.').resolve().parent   # assumes notebook lives in /notebooks/

# ── Directory structure ───────────────────────────────────────────────────────
DIRS = {
    'raw_viirs'  : PROJECT_ROOT / 'data' / 'raw' / 'viirs',
    'raw_firms'  : PROJECT_ROOT / 'data' / 'raw' / 'firms',
    'processed'  : PROJECT_ROOT / 'data' / 'processed',
    'masks'      : PROJECT_ROOT / 'data' / 'masks',
    'logs'       : PROJECT_ROOT / 'data' / 'raw' / '_logs',
}
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f'  {name:12s} → {path}')

print('\nDirectory structure ready.')

  raw_viirs    → C:\Users\taylo\firesat\data\raw\viirs
  raw_firms    → C:\Users\taylo\firesat\data\raw\firms
  processed    → C:\Users\taylo\firesat\data\processed
  masks        → C:\Users\taylo\firesat\data\masks
  logs         → C:\Users\taylo\firesat\data\raw\_logs

Directory structure ready.


In [4]:
# ── Spatial domain ────────────────────────────────────────────────────────────
# Western CONUS training domain
BBOX = {
    'lon_min': -125.0,
    'lon_max': -109.0,
    'lat_min':   32.0,
    'lat_max':   49.0,
}

# ── Temporal domain ───────────────────────────────────────────────────────────
TRAIN_YEARS = list(range(2018, 2023))   # 2018–2022 inclusive
VAL_YEAR    = 2023                       # held-out validation year
ALL_YEARS   = TRAIN_YEARS + [VAL_YEAR]

# Fire season months only (reduces download size by ~60%)
# Adjust if you later want year-round coverage
FIRE_SEASON_MONTHS = [5, 6, 7, 8, 9, 10]  # May–October

# ── VIIRS product configuration ───────────────────────────────────────────────
VIIRS_PRODUCT  = 'VNP14IMG'          # 375 m active fire, Suomi NPP
VIIRS_VERSION  = '001'
VIIRS_PLATFORM = 'SUOMI_NPP'         # also try 'NOAA_20' for VJ114IMG

# ── FIRMS configuration ───────────────────────────────────────────────────────
# FIRMS map key — register free at https://firms.modaps.eosdis.nasa.gov/api/
# Store your key here or as an environment variable FIRMS_MAP_KEY
FIRMS_MAP_KEY = '37ac8057666042849871ea1b5de95abb'
FIRMS_SOURCE  = 'VIIRS_SNPP_NRT'     # Near-real-time VIIRS Suomi NPP

print('Configuration set.')
print(f'  Training years : {TRAIN_YEARS}')
print(f'  Validation year: {VAL_YEAR}')
print(f'  Bounding box   : {BBOX}')
print(f'  Fire season    : months {FIRE_SEASON_MONTHS}')

Configuration set.
  Training years : [2018, 2019, 2020, 2021, 2022]
  Validation year: 2023
  Bounding box   : {'lon_min': -125.0, 'lon_max': -109.0, 'lat_min': 32.0, 'lat_max': 49.0}
  Fire season    : months [5, 6, 7, 8, 9, 10]


---
## Section 2 — Download FIRMS active fire ground truth

FIRMS is the faster download and gives us fire labels immediately. We pull it first so we can use it to guide the VIIRS spatial query.

**FIRMS API endpoint used:** `https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/{bbox}/{days}`

For multi-year historical data, FIRMS provides annual archive CSVs. We use those here.

> **Note:** The FIRMS map key is free. Register at https://firms.modaps.eosdis.nasa.gov/api/ — approval is instant.

In [5]:
def build_firms_archive_url(source: str, year: int) -> str:
    """
    Build the URL for a FIRMS annual archive CSV.

    FIRMS archive structure:
      https://firms.modaps.eosdis.nasa.gov/data/active_fire/{source_lower}/csv/
          {SOURCE}_{REGION}_{YEAR}.csv

    We use the global archive and filter spatially afterward — avoids
    dealing with FIRMS region codes which change between data versions.
    """
    source_lower = source.lower().replace('_nrt', '').replace('_sp', '')
    # FIRMS uses slightly different folder names per product
    folder_map = {
        'viirs_snpp': 'VIIRS_SNPP_NRT',
        'viirs_noaa20': 'VIIRS_NOAA20_NRT',
        'modis': 'MODIS_NRT',
    }
    folder = folder_map.get(source_lower, source)
    base = 'https://firms.modaps.eosdis.nasa.gov/data/active_fire'
    return f'{base}/{folder}/csv/{folder}_Global_{year}.csv'


def download_firms_year(
    year: int,
    source: str,
    bbox: dict,
    out_dir: Path,
    fire_months: list = None,
) -> Path:
    """
    Download one year of FIRMS archive CSV, filter to bounding box and
    fire-season months, and save a lean parquet file.

    Returns the path to the saved parquet file.
    """
    out_parquet = out_dir / f'firms_{source.lower()}_{year}.parquet'
    if out_parquet.exists():
        log.info(f'FIRMS {year} already exists — skipping download.')
        return out_parquet

    url = build_firms_archive_url(source, year)
    log.info(f'Downloading FIRMS {year}: {url}')

    try:
        df = pd.read_csv(url, low_memory=False)
    except Exception as e:
        log.error(f'Failed to download FIRMS {year}: {e}')
        log.error('Check your FIRMS map key and network connection.')
        raise

    log.info(f'  Raw rows: {len(df):,}')

    # ── Spatial filter ────────────────────────────────────────────────────
    df = df[
        (df['longitude'] >= bbox['lon_min']) &
        (df['longitude'] <= bbox['lon_max']) &
        (df['latitude']  >= bbox['lat_min']) &
        (df['latitude']  <= bbox['lat_max'])
    ].copy()
    log.info(f'  After bbox filter: {len(df):,}')

    # ── Parse acquisition date ────────────────────────────────────────────
    # FIRMS column is 'acq_date' (YYYY-MM-DD) and 'acq_time' (HHMM)
    df['acq_date'] = pd.to_datetime(df['acq_date'])
    df['month']    = df['acq_date'].dt.month

    # ── Temporal filter (fire season) ─────────────────────────────────────
    if fire_months:
        df = df[df['month'].isin(fire_months)].copy()
    log.info(f'  After month filter: {len(df):,}')

    # ── Keep operationally useful columns ─────────────────────────────────
    keep_cols = [
        'latitude', 'longitude', 'acq_date', 'acq_time',
        'brightness', 'bright_t31', 'frp',          # fire radiative power
        'confidence', 'satellite', 'instrument',
        'daynight', 'type',
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols]

    # ── Save ──────────────────────────────────────────────────────────────
    df.to_parquet(out_parquet, index=False)
    log.info(f'  Saved → {out_parquet} ({len(df):,} fire detections)')
    return out_parquet


print('FIRMS download functions defined.')

FIRMS download functions defined.


In [6]:
# ── Run FIRMS downloads ───────────────────────────────────────────────────────
firms_files = {}

for year in ALL_YEARS:
    print(f'\n── FIRMS {year} ──')
    try:
        fpath = download_firms_year(
            year=year,
            source=FIRMS_SOURCE,
            bbox=BBOX,
            out_dir=DIRS['raw_firms'],
            fire_months=FIRE_SEASON_MONTHS,
        )
        firms_files[year] = fpath
    except Exception as e:
        print(f'  ERROR on {year}: {e}')

print('\n── FIRMS download complete ──')
for yr, fp in firms_files.items():
    print(f'  {yr}: {fp.name}')

INFO | Downloading FIRMS 2018: https://firms.modaps.eosdis.nasa.gov/data/active_fire/VIIRS_SNPP_NRT/csv/VIIRS_SNPP_NRT_Global_2018.csv



── FIRMS 2018 ──


ERROR | Failed to download FIRMS 2018: HTTP Error 404: Not Found
ERROR | Check your FIRMS map key and network connection.
INFO | Downloading FIRMS 2019: https://firms.modaps.eosdis.nasa.gov/data/active_fire/VIIRS_SNPP_NRT/csv/VIIRS_SNPP_NRT_Global_2019.csv


  ERROR on 2018: HTTP Error 404: Not Found

── FIRMS 2019 ──


ERROR | Failed to download FIRMS 2019: HTTP Error 404: Not Found
ERROR | Check your FIRMS map key and network connection.
INFO | Downloading FIRMS 2020: https://firms.modaps.eosdis.nasa.gov/data/active_fire/VIIRS_SNPP_NRT/csv/VIIRS_SNPP_NRT_Global_2020.csv


...

  ERROR on 2021: HTTP Error 404: Not Found

── FIRMS 2022 ──


ERROR | Failed to download FIRMS 2022: HTTP Error 404: Not Found
ERROR | Check your FIRMS map key and network connection.
INFO | Downloading FIRMS 2023: https://firms.modaps.eosdis.nasa.gov/data/active_fire/VIIRS_SNPP_NRT/csv/VIIRS_SNPP_NRT_Global_2023.csv


  ERROR on 2022: HTTP Error 404: Not Found

── FIRMS 2023 ──


ERROR | Failed to download FIRMS 2023: HTTP Error 404: Not Found
ERROR | Check your FIRMS map key and network connection.


  ERROR on 2023: HTTP Error 404: Not Found

── FIRMS download complete ──


In [8]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Install required packages
!pip install earthaccess requests pandas numpy h5py xarray tqdm -q

# Verify
import earthaccess
print('earthaccess:', earthaccess.__version__)

ModuleNotFoundError: No module named 'google.colab'

In [7]:
# ── Quick FIRMS summary ───────────────────────────────────────────────────────
firms_all = pd.concat(
    [pd.read_parquet(fp) for fp in firms_files.values()],
    ignore_index=True
)

print('=== FIRMS Summary ===')
print(f'Total fire detections  : {len(firms_all):>10,}')
print(f'Date range             : {firms_all["acq_date"].min().date()} → {firms_all["acq_date"].max().date()}')
print(f'Longitude range        : {firms_all["longitude"].min():.2f}° → {firms_all["longitude"].max():.2f}°')
print(f'Latitude range         : {firms_all["latitude"].min():.2f}° → {firms_all["latitude"].max():.2f}°')
print(f'\nFire Radiative Power (FRP) stats (MW):')
print(firms_all['frp'].describe().to_string())
print(f'\nDetections by year:')
firms_all['acq_date'] = pd.to_datetime(firms_all['acq_date'])
print(firms_all.groupby(firms_all['acq_date'].dt.year).size().rename('count').to_string())

ValueError: No objects to concatenate

---
## Section 3 — Download VIIRS VNP14IMG (375 m active fire product)

**VNP14IMG** is the VIIRS 375 m active fire product from Suomi NPP. It is derived from the I4 (3.74 µm MWIR) and I5 (11.45 µm LWIR) bands — exactly the FireSat-analog channels.

We use the **NASA LAADS DAAC** REST API. Each granule is one ~5-minute orbital swath segment in HDF5 format.

**Download strategy:** Rather than downloading all VIIRS granules (many TB), we use FIRMS locations to identify the specific overpasses we need, then download only those granules that overlap our domain and contain confirmed fire detections. This reduces data volume by ~90%.

**LAADS API documentation:** https://ladsweb.modaps.eosdis.nasa.gov/tools-and-services/data-download-scripts/

In [ ]:
# ── LAADS DAAC API helper functions ───────────────────────────────────────────

LAADS_BASE = 'https://ladsweb.modaps.eosdis.nasa.gov'

def laads_search_granules(
    product: str,
    version: str,
    start_date: str,
    end_date: str,
    bbox: dict,
    session: requests.Session,
) -> list:
    """
    Search LAADS DAAC for granules matching product, date range, and bbox.

    Returns a list of granule dicts with at minimum:
        {'fileId': ..., 'fileName': ..., 'fileSize': ..., 'downloadUrl': ...}
    """
    url = f'{LAADS_BASE}/api/v1/files/product'
    params = {
        'products'   : product,
        'collection' : version,
        'startTime'  : start_date,
        'endTime'    : end_date,
        'south'      : bbox['lat_min'],
        'north'      : bbox['lat_max'],
        'west'       : bbox['lon_min'],
        'east'       : bbox['lon_max'],
        'daacDatasetFields': 'fileName,fileSize',
    }

    resp = session.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    # LAADS returns a flat list of file objects
    granules = []
    for item in data:
        granules.append({
            'fileName'   : item.get('fileName', ''),
            'fileSize'   : item.get('fileSize', 0),
            'downloadUrl': f"{LAADS_BASE}/archive/allData/{version}/{product}/{item.get('filePath', '')}",
        })
    return granules


def download_granule(
    url: str,
    out_path: Path,
    session: requests.Session,
    retries: int = 3,
    chunk_size: int = 1024 * 1024,  # 1 MB chunks
) -> bool:
    """
    Download a single granule to out_path. Returns True on success.
    Uses streaming download with retry logic.
    """
    if out_path.exists() and out_path.stat().st_size > 10_000:
        return True  # Already downloaded and non-trivial size

    for attempt in range(retries):
        try:
            with session.get(url, stream=True, timeout=120) as resp:
                resp.raise_for_status()
                with open(out_path, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=chunk_size):
                        if chunk:
                            f.write(chunk)
            return True
        except Exception as e:
            log.warning(f'Attempt {attempt+1}/{retries} failed for {out_path.name}: {e}')
            time.sleep(2 ** attempt)  # exponential backoff
            if out_path.exists():
                out_path.unlink()   # remove partial file

    log.error(f'All {retries} attempts failed for {url}')
    return False


print('LAADS download functions defined.')

In [ ]:
# ── Build FIRMS-guided date list for VIIRS downloads ─────────────────────────
# Strategy: for each day in FIRMS that has ≥ MIN_FIRE_COUNT detections in
# our domain, add it to the VIIRS download queue.
# This keeps downloads focused on days with actual fire activity.

MIN_FIRE_COUNT = 5   # minimum FIRMS detections in domain to trigger VIIRS download

firms_all['date_str'] = firms_all['acq_date'].dt.strftime('%Y-%m-%d')
fire_days = (
    firms_all
    .groupby('date_str')
    .size()
    .rename('count')
    .reset_index()
    .query(f'count >= {MIN_FIRE_COUNT}')
    .sort_values('date_str')
)

print(f'Days with ≥{MIN_FIRE_COUNT} FIRMS detections in domain: {len(fire_days)}')
print(f'Earliest: {fire_days["date_str"].iloc[0]}')
print(f'Latest:   {fire_days["date_str"].iloc[-1]}')
print(f'\nTop 10 busiest fire days:')
print(fire_days.nlargest(10, 'count').to_string(index=False))

In [ ]:
# ── Authenticated session for LAADS ──────────────────────────────────────────
# NASA Earthdata uses .netrc for authentication with requests

import netrc

def build_earthdata_session() -> requests.Session:
    """
    Create a requests.Session with NASA Earthdata credentials.
    Reads from ~/.netrc (set up in Section 0).
    """
    try:
        creds = netrc.netrc()
        auth  = creds.authenticators('urs.earthdata.nasa.gov')
        if auth is None:
            raise ValueError('No urs.earthdata.nasa.gov entry in ~/.netrc')
        username, _, password = auth
    except FileNotFoundError:
        raise FileNotFoundError(
            '~/.netrc not found. Run Section 0 to set up credentials.'
        )

    session = requests.Session()
    session.auth = (username, password)

    # Follow redirects (NASA uses 302 redirect to S3)
    session.max_redirects = 10

    # Test authentication
    test_url = 'https://urs.earthdata.nasa.gov/api/users/tokens'
    resp = session.get(test_url, timeout=15)
    if resp.status_code == 200:
        print('NASA Earthdata authentication: OK')
    else:
        print(f'WARNING: Auth test returned {resp.status_code}. Proceeding anyway.')

    return session


session = build_earthdata_session()

In [ ]:
# ── Search LAADS for available VNP14IMG granules ──────────────────────────────
# We query by month-blocks to keep the API response sizes manageable.
# LAADS has rate limits — add small delays between requests.

def month_date_ranges(years: list, months: list):
    """Generate (start_date_str, end_date_str) pairs for each year-month combo."""
    import calendar
    for year in years:
        for month in months:
            _, last_day = calendar.monthrange(year, month)
            yield (
                f'{year}-{month:02d}-01',
                f'{year}-{month:02d}-{last_day:02d}',
            )


all_granules = []

date_ranges = list(month_date_ranges(ALL_YEARS, FIRE_SEASON_MONTHS))
print(f'Querying LAADS for {len(date_ranges)} month-blocks...')

for start, end in tqdm(date_ranges, desc='LAADS search'):
    try:
        granules = laads_search_granules(
            product    = VIIRS_PRODUCT,
            version    = VIIRS_VERSION,
            start_date = start,
            end_date   = end,
            bbox       = BBOX,
            session    = session,
        )
        for g in granules:
            g['query_start'] = start
            g['query_end']   = end
        all_granules.extend(granules)
        time.sleep(0.5)  # be polite to LAADS API
    except Exception as e:
        log.warning(f'Search failed for {start}–{end}: {e}')

print(f'\nTotal granules found: {len(all_granules):,}')

# Deduplicate by filename
seen = set()
unique_granules = []
for g in all_granules:
    if g['fileName'] not in seen:
        seen.add(g['fileName'])
        unique_granules.append(g)

print(f'Unique granules after dedup: {len(unique_granules):,}')
total_gb = sum(g.get('fileSize', 0) for g in unique_granules) / 1e9
print(f'Estimated download size: {total_gb:.1f} GB')

In [ ]:
# ── Save granule manifest ─────────────────────────────────────────────────────
manifest_path = DIRS['logs'] / 'viirs_granule_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(unique_granules, f, indent=2)
print(f'Granule manifest saved: {manifest_path}')

# ── Filter to FIRMS-active days only ─────────────────────────────────────────
# VNP14IMG filename format: VNP14IMG.AYYYYDDD.hhmm.001.*.h5
# where YYYY = year, DDD = day-of-year
# We match against the fire_days date strings

active_date_strs = set(fire_days['date_str'].tolist())

def granule_date_str(filename: str) -> str:
    """Extract YYYY-MM-DD from VNP14IMG filename."""
    # e.g. VNP14IMG.A2020244.0000.001.2020244070707.h5
    m = re.search(r'\.A(\d{7})\.', filename)
    if not m:
        return ''
    year_doy = m.group(1)
    year = int(year_doy[:4])
    doy  = int(year_doy[4:])
    return (date(year, 1, 1) + timedelta(days=doy - 1)).strftime('%Y-%m-%d')


filtered_granules = [
    g for g in unique_granules
    if granule_date_str(g['fileName']) in active_date_strs
]

print(f'Granules on FIRMS-active days: {len(filtered_granules):,}')
filtered_gb = sum(g.get('fileSize', 0) for g in filtered_granules) / 1e9
print(f'Filtered download size: {filtered_gb:.1f} GB  (vs {total_gb:.1f} GB total)')

In [ ]:
# ── Download VIIRS granules ───────────────────────────────────────────────────
# Downloads run with a thread pool for speed. Adjust MAX_WORKERS to suit
# your connection — 4 is safe; 8 is fine if you have a fast link.
#
# On a slow connection, set DRY_RUN = True first to verify the manifest,
# then set it to False to actually download.

DRY_RUN     = True   # ← set to False when ready to download
MAX_WORKERS = 4

if DRY_RUN:
    print('DRY RUN mode — no files will be downloaded.')
    print(f'Would download {len(filtered_granules)} granules (~{filtered_gb:.1f} GB)')
    print('Set DRY_RUN = False to begin.')
else:
    success_count = 0
    fail_count    = 0
    failed_granules = []

    def _download_task(granule):
        fname    = granule['fileName']
        out_path = DIRS['raw_viirs'] / fname
        ok = download_granule(
            url      = granule['downloadUrl'],
            out_path = out_path,
            session  = session,
        )
        return fname, ok

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_download_task, g): g for g in filtered_granules}
        with tqdm(total=len(futures), desc='VIIRS download') as pbar:
            for future in as_completed(futures):
                fname, ok = future.result()
                if ok:
                    success_count += 1
                else:
                    fail_count += 1
                    failed_granules.append(fname)
                pbar.update(1)
                pbar.set_postfix({'ok': success_count, 'fail': fail_count})

    print(f'\nDownload complete: {success_count} OK, {fail_count} failed')
    if failed_granules:
        fail_log = DIRS['logs'] / 'failed_granules.txt'
        fail_log.write_text('\n'.join(failed_granules))
        print(f'Failed granule list saved: {fail_log}')

---
## Section 4 — Inspect a downloaded VNP14IMG granule

Before moving on, verify the HDF5 structure of a downloaded file. This confirms the bands are present and the data types are correct for Module 1c (harmonization).

In [ ]:
# ── HDF5 structure inspection ─────────────────────────────────────────────────
if not HDF5_AVAILABLE:
    print('h5py not available — skipping inspection.')
else:
    viirs_files = sorted(DIRS['raw_viirs'].glob('*.h5'))

    if not viirs_files:
        print('No VIIRS HDF5 files found yet.')
        print('Complete the download in Section 3 (set DRY_RUN = False) then rerun.')
    else:
        sample_file = viirs_files[0]
        print(f'Inspecting: {sample_file.name}\n')

        def print_hdf5_tree(name, obj):
            indent = '  ' * name.count('/')
            if hasattr(obj, 'shape'):
                dtype_str = str(obj.dtype)
                print(f'{indent}{name.split("/")[-1]}  shape={obj.shape}  dtype={dtype_str}')
            else:
                print(f'{indent}[{name.split("/")[-1]}]')

        with h5py.File(sample_file, 'r') as f:
            print('=== HDF5 Groups and Datasets ===')
            f.visititems(print_hdf5_tree)

            print('\n=== Key variable ranges ===')
            # VNP14IMG standard variable names
            fire_vars = [
                'Fire Pixels/FP_latitude',
                'Fire Pixels/FP_longitude',
                'Fire Pixels/FP_T4',       # brightness temp I4 (MWIR)
                'Fire Pixels/FP_T5',       # brightness temp I5 (LWIR)
                'Fire Pixels/FP_power',    # fire radiative power (MW)
                'Fire Pixels/FP_confidence',
            ]
            for var in fire_vars:
                if var in f:
                    data = f[var][:]
                    print(f'  {var.split("/")[-1]:20s}  min={data.min():.2f}  max={data.max():.2f}  n={len(data)}')

---
## Section 5 — Archive integrity check and summary

Final cell: count files, estimate coverage, flag any gaps. This output becomes the starting point for Module 1b (ERA5/AOD downloads).

In [ ]:
# ── Archive summary ───────────────────────────────────────────────────────────
viirs_files  = sorted(DIRS['raw_viirs'].glob('*.h5'))
firms_files_ = sorted(DIRS['raw_firms'].glob('*.parquet'))

total_viirs_gb = sum(f.stat().st_size for f in viirs_files) / 1e9
total_firms_mb = sum(f.stat().st_size for f in firms_files_) / 1e6

print('=' * 55)
print('  FireSight-IR | Module 1a Complete')
print('=' * 55)
print(f'  VIIRS HDF5 granules : {len(viirs_files):>6,}  ({total_viirs_gb:.2f} GB)')
print(f'  FIRMS parquet files : {len(firms_files_):>6,}  ({total_firms_mb:.1f} MB)')
print()
print('  Paths:')
print(f'    VIIRS → {DIRS["raw_viirs"]}')
print(f'    FIRMS → {DIRS["raw_firms"]}')
print()
print('  Next notebook: 01b_download_era5_aod.ipynb')
print('    → Downloads ERA5 atmospheric profiles and MODIS AOD')
print('    → Co-locates atmospheric columns with FIRMS fire pixels')
print('=' * 55)

# ── Save run log ──────────────────────────────────────────────────────────────
run_log = {
    'notebook'         : '01a_download_viirs_firms',
    'run_date'         : str(date.today()),
    'viirs_granules'   : len(viirs_files),
    'firms_years'      : list(firms_files.keys()),
    'total_firms_detections': int(len(firms_all)),
    'bbox'             : BBOX,
    'train_years'      : TRAIN_YEARS,
    'val_year'         : VAL_YEAR,
    'fire_months'      : FIRE_SEASON_MONTHS,
}
(DIRS['logs'] / '01a_run_log.json').write_text(json.dumps(run_log, indent=2))
print('Run log saved.')

---
## Notes for Module 1b

Take the following into the next notebook:

- **`firms_all` DataFrame** — the full fire detection table for the domain. Module 1b uses the unique `(date, lat, lon)` tuples to pull ERA5 columns at exactly those locations and times.
- **`fire_days`** — the date list drives ERA5 download batching.
- **VIIRS granule manifest** at `data/raw/_logs/viirs_granule_manifest.json` — used in Module 1c to match granules to ERA5 timestamps for co-registration.

### Troubleshooting common issues

| Issue | Likely cause | Fix |
|---|---|---|
| `401 Unauthorized` on LAADS | `.netrc` not written correctly | Rerun Section 0, verify `~/.netrc` contents |
| `FIRMS download returns empty DataFrame` | FIRMS map key missing or invalid | Set `FIRMS_MAP_KEY` env var or paste key in config cell |
| `granule_date_str` returns empty string | Filename format differs (NOAA-20 vs Suomi NPP) | Inspect filename and adjust regex in `granule_date_str()` |
| Very large granule count | Bbox too wide or version mismatch | Confirm `VIIRS_VERSION = '001'` and recheck bbox |
| Download hangs | LAADS server timeout | Reduce `MAX_WORKERS` to 2 and add `time.sleep(1)` in task loop |